In [14]:
import pandas as pd

df_2010 = pd.read_csv('dados_brutos/faixa_etaria_ibge_2010.csv', sep=';', skiprows=4, skipfooter=1, engine='python')
df_2010

,Cód.,Unidade da Federação e Município,Sexo,Idade,Total
0,Município (Código),Unidade da Federação e Município,Sexo,Idade,Total
1,15,Pará,Total,Total,7581051
2,15,Pará,Total,0 a 4 anos,736655
3,15,Pará,Total,Menos de 1 ano,142182
4,15,Pará,Total,Menos de 1 mês,10995
...,...,...,...,...,...
19590,-,"Zero absoluto, não resultante de um cálculo ou...",NaN,NaN,NaN
19591,0,Zero resultante de um cálculo ou arredondament...,NaN,NaN,NaN
19592,X,Valor inibido para não identificar o informant...,NaN,NaN,NaN
19593,..,Valor não se aplica.\r\nEx: Não se pode obter ...,NaN,NaN,NaN


In [17]:
import pandas as pd
import re


# 2. Renomear colunas de identificação para o padrão desejado
# Ajuste os nomes das colunas originais se estiverem diferentes no seu arquivo
colunas_id = {
    'Cód.': 'Codigo', 
    'Unidade da Federação e Município': 'Municipio'
}
df_2010 = df_2010.rename(columns=colunas_id)

# Remover a sigla do estado do nome do município (ex: "Abaetetuba (PA)" -> "ABAETETUBA")
if 'Municipio' in df_2010.columns:
    df_2010['Municipio'] = df_2010['Municipio'].str.split(' \(').str[0].str.upper()
    df_2010['Codigo'] = df_2010['Codigo'].astype(str).str[:6] # Manter 6 dígitos do IBGE

# 3. Função para mapear as colunas originais para as novas faixas
def classificar_faixa(nome_coluna):
    # Busca números na string da coluna
    numeros = [int(n) for n in re.findall(r'\d+', str(nome_coluna))]
    
    if not numeros or 'Total' in str(nome_coluna):
        return 'Total'
    
    idade = numeros[0]
    
    # Se a coluna for "65 anos ou mais", cai no 64+
    if 'mais' in str(nome_coluna).lower() or idade >= 64:
        return '64+'
    
    # Classificação nas faixas
    if 0 <= idade <= 3: return '0-3'
    elif 4 <= idade <= 6: return '4-6'
    elif 7 <= idade <= 15: return '7-15'
    elif 16 <= idade <= 17: return '16-17'
    elif 18 <= idade <= 24: return '18-24'
    elif 25 <= idade <= 34: return '25-34'
    elif 35 <= idade <= 39: return '35-39'
    elif 40 <= idade <= 44: return '40-44'
    elif 45 <= idade <= 49: return '45-49'
    elif 50 <= idade <= 54: return '50-54'
    elif 55 <= idade <= 59: return '55-59'
    elif 60 <= idade <= 64: return '60-64'
    
    return None

# 4. Derreter (Melt) a tabela para facilitar o agrupamento
df_melt = df_2010.melt(id_vars=['Codigo', 'Municipio'], var_name='Idade_Original', value_name='Populacao')

# Limpar os dados populacionais (remover hífens, NAs e converter para float)
df_melt['Populacao'] = pd.to_numeric(df_melt['Populacao'].replace('-', 0).replace('X', 0), errors='coerce').fillna(0)

# Aplicar o mapeamento de faixas etárias
df_melt['Faixa_Agrupada'] = df_melt['Idade_Original'].apply(classificar_faixa)

# Filtrar apenas as linhas que receberam uma classificação válida
df_melt = df_melt.dropna(subset=['Faixa_Agrupada'])

# 5. Pivotar a tabela de volta para o formato de colunas (Wide)
df_final_2010 = df_melt.pivot_table(
    index=['Codigo', 'Municipio'], 
    columns='Faixa_Agrupada', 
    values='Populacao', 
    aggfunc='sum'
).reset_index()

# 6. Reordenar as colunas na estrutura exata solicitada
ordem_colunas = ['Codigo', 'Municipio', '0-3', '4-6', '7-15', '16-17', '18-24', 
                 '25-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '64+', 'Total']

# Adicionar colunas faltantes com NaN caso alguma faixa não exista nos dados originais
for col in ordem_colunas:
    if col not in df_final_2010.columns:
        df_final_2010[col] = float('nan')

df_final_2010 = df_final_2010[ordem_colunas]


df_final_2010

<>:15: SyntaxWarning: invalid escape sequence '\('
<>:15: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_15660/541417169.py:15: SyntaxWarning: invalid escape sequence '\('
  df_2010['Municipio'] = df_2010['Municipio'].str.split(' \(').str[0].str.upper()


Faixa_Agrupada,Codigo,Municipio,0-3,4-6,7-15,16-17,18-24,25-34,35-39,40-44,45-49,50-54,55-59,60-64,64+,Total
0,-,"ZERO ABSOLUTO, NÃO RESULTANTE DE UM CÁLCULO OU...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,..,VALOR NÃO SE APLICA.\r\nEX: NÃO SE PODE OBTER ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,...,VALOR NÃO DISPONÍVEL.\r\nEX: A PRODUÇÃO DE FEI...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,0,ZERO RESULTANTE DE UM CÁLCULO OU ARREDONDAMENT...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,15,PARÁ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24206749.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,150835,VITÓRIA DO XINGU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42797.0
147,150840,XINGUARA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,129085.0
148,Municí,UNIDADE DA FEDERAÇÃO E MUNICÍPIO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
149,Símbol,SIGNIFICADO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [ ]:
df_final_2010.to_csv('dados_tratados/faixa_etaria_ibge.csv',index=False)